# Environment configuration — bronze/hrm
Credentials are external. Set PIPELINE_ENV and job variables generated from config/environments and config/secrets.
This repository revision changes configuration handling, not the source ingestion/write algorithms.


# NB_bronze.hrm.unit — batch upsert theo ngày
Giữ cấu hình kết nối S3/metastore/Spark từ notebook mẫu. Không cần các cờ confirm.
API trả các bản ghi upsert trong khoảng request: date_start=ETL_Date, date_end=ETL_Date+1 ngày.
Mặc định ETL_Date=T−1 theo giờ Việt Nam; data_date giữ YYYY-MM-DD.
Landing: `s3://landing-bucket/hrm/unit/YYYYMMDD/page-000001.json` và `manifest.json`.
Mỗi ngày chỉ có một tập file hiện hành, không có thư mục runs. RUN_ID vẫn lưu trong manifest/log để truy vết.


In [ ]:
# Như mẫu: dừng SparkSession cũ trước khi khởi tạo lại cấu hình.
# Chạy cell này một lần khi bắt đầu notebook, không chạy giữa lúc job đang ghi.
if "spark" in globals() and spark is not None:
    spark.stop()


In [ ]:
import logging, time, os, json, ast, traceback, re, uuid, hashlib, sys
from datetime import datetime, timedelta, date, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import boto3
from botocore.config import Config
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F, types as T
from pyspark.sql.types import *


In [ ]:
# Log Setting
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True 
)
logger = logging.getLogger(__name__)

In [ ]:
# start_date = datetime.now() + timedelta(hours=7)
start_date = datetime.now()
log_message = None
status = "Success"
record_count = 0
df = None

In [ ]:
# Environment configuration is injected by CP4D/job environment variables.
# Generate a job env file with scripts/build_job_env.py. Never embed credentials here.
def _need_env(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu biến môi trường: " + name)
    return value

PIPELINE_ENV = _need_env("PIPELINE_ENV")
if PIPELINE_ENV not in ("dev", "uat", "prod"):
    raise ValueError("PIPELINE_ENV phải là dev, uat hoặc prod")

# 1. Lấy các biến môi trường
now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
yesterday = now - timedelta(days=1)

# 3. Định dạng lại thành chuỗi YYYYMMDD
yesterday = yesterday.strftime("%Y-%m-%d")

#S3
S3_ENDPOINT = _need_env("S3_ENDPOINT")
ACCESS_KEY = _need_env("ACCESS_KEY")
SECRET_KEY = _need_env("SECRET_KEY")
BUCKET_NAME = _need_env("BUCKET_NAME")

#Metastore
WXD_HMS_USERNAME = _need_env("WXD_HMS_USERNAME")
WXD_HMS_PASSWORD = _need_env("WXD_HMS_PASSWORD")
ICEBERG_CATALOG1 = os.getenv("ICEBERG_CATALOG1", "bronze")
BUCKET_NAME1 = _need_env("BUCKET_NAME1")
ACCESS_KEY1 = os.getenv("ACCESS_KEY1") or ACCESS_KEY
SECRET_KEY1 = os.getenv("SECRET_KEY1") or SECRET_KEY

HIVEMETASTORE_HOST = _need_env("HIVEMETASTORE_HOST")

HIVE_TRUSTSTORE_PATH = _need_env("HIVE_TRUSTSTORE_PATH")
HIVE_TRUSTSTORE_PASSWORD = _need_env("HIVE_TRUSTSTORE_PASSWORD")

WXD_INSTANCE_ID = _need_env("WXD_INSTANCE_ID")
HADOOP_AMS_URL = _need_env("HADOOP_AMS_URL")
HADOOP_CAS_ENDPOINT = _need_env("HADOOP_CAS_ENDPOINT")
HADOOP_CPG_ENDPOINT = _need_env("HADOOP_CPG_ENDPOINT")


TABLE_NAME=os.getenv("TABLE_NAME","bronze.hrm.unit")
SOURCE_NAME=os.getenv("SOURCE_NAME","hrm.api.unit")
STEP='landing_2_bronze'


ETL_Date = os.getenv("ETL_Date",yesterday)

# HRM: entity và tham số chạy
ENTITY = 'unit'
LAYER = 'bronze'
LOG_TABLE = os.getenv("LOG_TABLE", "bronze.pipeline.log_pipeline")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", ICEBERG_CATALOG1):
    raise ValueError("ICEBERG_CATALOG1 không hợp lệ")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", TABLE_NAME):
    raise ValueError("TABLE_NAME phải là catalog.schema.table hợp lệ")
# Truyền ETL_Date qua job hoặc sửa dòng sau trước khi chạy các cell tiếp theo:
# ETL_Date = "2026-10-20"

# Copy đúng manifest từ notebook Landing cùng entity/ETL_Date.
MANIFEST_PATH_CONFIG = ""
LANDING_RUN_ID_CONFIG = ""
if MANIFEST_PATH_CONFIG: os.environ.setdefault("MANIFEST_PATH", MANIFEST_PATH_CONFIG)
if LANDING_RUN_ID_CONFIG: os.environ.setdefault("LANDING_RUN_ID", LANDING_RUN_ID_CONFIG)
SOURCE_NAME = os.getenv("SOURCE_NAME", f"s3://{BUCKET_NAME}/hrm/{ENTITY}/{datetime.strptime(ETL_Date, '%Y-%m-%d').strftime('%Y%m%d')}")


In [ ]:
def init_spark():
    spark = SparkSession.builder \
        .appName(f"import_table") \
        .config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}","org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.type", "hive") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.uri",
                f"{HIVEMETASTORE_HOST}?catalog={ICEBERG_CATALOG1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.warehouse",
                f"s3a://{BUCKET_NAME1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.username",
                WXD_HMS_USERNAME) \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.password",
                WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.uris",HIVEMETASTORE_HOST) \
        .config("spark.hive.metastore.use.SSL", "true") \
        .config("spark.hive.metastore.truststore.type", "JKS") \
        .config("spark.hive.metastore.truststore.path", HIVE_TRUSTSTORE_PATH) \
        .config("spark.hive.metastore.truststore.password", HIVE_TRUSTSTORE_PASSWORD) \
        .config("spark.hive.metastore.client.auth.mode", "PLAIN") \
        .config("spark.hive.metastore.client.plain.username", WXD_HMS_USERNAME) \
        .config("spark.hive.metastore.client.plain.password", WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.client.thrift.transport.mode", "http") \
        .config("spark.hadoop.wxd.ams.url", HADOOP_AMS_URL) \
        .config("spark.hadoop.wxd.cas.endpoint", HADOOP_CAS_ENDPOINT) \
        .config("spark.hadoop.wxd.cpg.endpoint", HADOOP_CPG_ENDPOINT) \
        .config("spark.hadoop.wxd.instanceId", WXD_INSTANCE_ID) \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        \
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.access.key", ACCESS_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.secret.key", SECRET_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.connection.ssl.enabled", "true") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.access.key", ACCESS_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.secret.key", SECRET_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.connection.ssl.enabled", "true") \
        \
        .config("spark.hadoop.iceberg.engine.hive.lock-enabled", "false") \
        .config("spark.sql.catalogImplementation", "hive") \
        .getOrCreate()


    spark.sparkContext.setLogLevel("WARN")
    return spark

In [ ]:
def required(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu cấu hình: " + name)
    return value

BATCH_DATE = date.fromisoformat(ETL_Date)
if BATCH_DATE.isoformat() != ETL_Date:
    raise ValueError("ETL_Date phải đúng YYYY-MM-DD")
RUN_ID = uuid.uuid4().hex
ALLOW_EMPTY = True  # API success với 0 upsert là batch rỗng hợp lệ.
DAY_PREFIX = f"hrm/{ENTITY}/{BATCH_DATE.strftime('%Y%m%d')}"
LOG_PREFIX = f"hrm/{ENTITY}/_logs/{BATCH_DATE.strftime('%Y%m%d')}"
SOURCE = SOURCE_NAME

def pack(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")

def prefix(run_id):
    if not re.fullmatch(r"[0-9a-f]{32}", run_id):
        raise ValueError("run_id không hợp lệ")
    return DAY_PREFIX

def init_landing():
    # Reuse exactly S3_ENDPOINT, ACCESS_KEY, SECRET_KEY, BUCKET_NAME from template.
    if not SECRET_KEY or SECRET_KEY.startswith("{dsnextenc}"):
        raise ValueError("SECRET_KEY phải là secret S3 dạng gốc như file mẫu")
    return boto3.client("s3", endpoint_url=S3_ENDPOINT,
        aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4", connect_timeout=15, read_timeout=60,
                      retries={"max_attempts": 3, "mode": "standard"},
                      s3={"addressing_style": "path"}))

def put_json(key, obj):
    blob = pack(obj)
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=blob, ContentType="application/json")
    return hashlib.sha256(blob).hexdigest()

def get_json(key, digest=None):
    blob = s3.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read()
    if digest and hashlib.sha256(blob).hexdigest() != digest:
        raise ValueError("Landing checksum không khớp")
    return json.loads(blob)

def error_summary(exc):
    # Các ValueError do notebook tạo không chứa token hay nội dung nhân sự.
    if isinstance(exc, ValueError) and not isinstance(exc, json.JSONDecodeError):
        return str(exc)
    response = getattr(exc, "response", None)
    if isinstance(response, dict):
        return "S3 error: " + str(response.get("Error", {}).get("Code", "unknown"))
    return type(exc).__name__ + ": kiểm tra kết nối/runtime; không ghi response hoặc credentials."


In [ ]:
def resolve_manifest():
    expected_key = f"{DAY_PREFIX}/manifest.json"
    path = os.getenv("MANIFEST_PATH", "").strip()
    if path:
        parsed = urlparse(path)
        if (parsed.scheme != "s3" or parsed.netloc != BUCKET_NAME or
                parsed.path.lstrip("/") != expected_key or parsed.query or parsed.fragment):
            raise ValueError("MANIFEST_PATH phải trỏ đúng hrm/<entity>/YYYYMMDD/manifest.json của ETL_Date")
    manifest = get_json(expected_key)
    run_id = os.getenv("LANDING_RUN_ID", "").strip()
    if run_id and manifest.get("run_id") != run_id:
        raise ValueError("Landing ngày đã được tạo lại: LANDING_RUN_ID không khớp manifest hiện tại")
    if manifest.get("manifest_version") != 3:
        raise ValueError("Cần manifest version 3 của cấu trúc thư mục YYYYMMDD")
    return manifest

def load_landing(manifest):
    if manifest.get("complete") is not True or manifest.get("entity") != ENTITY or manifest.get("data_date") != ETL_Date:
        raise ValueError("Manifest chưa chốt hoặc sai entity/ngày")
    if type(manifest.get("count")) is not int or manifest["count"] < 0:
        raise ValueError("Manifest count không hợp lệ")
    files = manifest.get("files")
    if not isinstance(files, list) or not files:
        raise ValueError("Manifest không có danh sách trang")
    rows, seen = [], set()
    for page, item in enumerate(files, 1):
        if item.get("key") != f"{prefix(manifest['run_id'])}/page-{page:06d}.json":
            raise ValueError("Manifest sai thứ tự/đường dẫn trang")
        if not re.fullmatch(r"[0-9a-f]{64}", item.get("sha256", "")):
            raise ValueError("Thiếu checksum trang")
        payload = get_json(item["key"], item["sha256"])
        part = payload.get("result")
        if payload.get("success") is not True or not isinstance(part, list) or len(part) != item.get("count"):
            raise ValueError("Landing page không hợp lệ")
        for row in part:
            if not isinstance(row, dict) or row.get("id") is None or str(row["id"]) in seen:
                raise ValueError("Landing có id thiếu/trùng")
            seen.add(str(row["id"]))
        rows.extend(part)
    if len(rows) != manifest["count"]:
        raise ValueError("Số dòng Landing không khớp manifest")
    if not rows and not ALLOW_EMPTY:
        raise ValueError("Batch rỗng: chỉ đặt HRM_ALLOW_EMPTY=true sau khi xác nhận")
    return rows

BUSINESS_FIELDS = {'unit': ['id', 'parent_id', 'code', 'name', 'full_path', 'parent_code', 'create_date', 'write_date', 'state', 'buudientinh_id', 'buudientinh_code', 'action', 'status', 'level', 'type', 'location_code', 'mohinhhoatdong', 'diabantinhtp', 'diabanxaphuong', 'diachichitiet', 'tenlienhe', 'dienthoai', 'thoigianlamviec', 'kinhdo', 'vido', 'type_code', 'type_name'], 'employee': ['id', 'employee_code', 'employee_name', 'birthday', 'sex', 'quequan', 'tamtru', 'quoctich', 'dantoc', 'tongiao', 'cccd', 'ngaycap_cccd', 'noicap_cccd', 'mobile_phone', 'work_phone', 'work_email', 'bank_account_number', 'bank_name', 'masothue', 'job_id', 'job_name', 'position_id', 'position_name', 'chucdanhkiemnhiem_ids', 'department_id', 'department_code', 'ngayvao_tct', 'ngayvao_tct_chinhthuc', 'ngayvao_donvi', 'ngayvao_donvi_chinhthuc', 'create_date', 'write_date', 'status', 'ngaynghiviec', 'action']}

def as_string(v):
    if v is None or isinstance(v, str):
        return v
    if isinstance(v, (dict, list, bool)):
        return pack(v).decode("utf-8")
    return str(v)

def bronze_frame(rows, manifest):
    names = BUSINESS_FIELDS[ENTITY]
    schema = T.StructType([T.StructField(n, T.StringType(), True) for n in names] + [
        T.StructField("raw_record", T.StringType(), False),
        T.StructField("data_date", T.DateType(), False),
        T.StructField("ingest_time", T.TimestampType(), False),
        T.StructField("run_id", T.StringType(), False),
        T.StructField("landing_run_id", T.StringType(), False)])
    stamp = datetime.fromisoformat(manifest["ingest_time"]).astimezone(timezone.utc)
    records, seen = [], set()
    for row in rows:
        key = str(row.get("id"))
        if row.get("id") is None or key in seen:
            raise ValueError("id thiếu/trùng trong Landing")
        seen.add(key)
        records.append(tuple(as_string(row.get(n)) for n in names) +
                       (pack(row).decode("utf-8"), BATCH_DATE, stamp, RUN_ID, manifest["run_id"]))
    return spark.createDataFrame(records, schema)

def write_bronze(frame):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.hrm")
    ddl = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in frame.schema)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {TABLE} ({ddl}) USING iceberg "
              "PARTITIONED BY (data_date) TBLPROPERTIES ('write.target-file-size-bytes'='134217728', 'write.distribution-mode'='none')")
    if spark.table(TABLE).schema.simpleString() != frame.schema.simpleString():
        raise ValueError("Schema Bronze khác cấu hình; cần review migration")
    frame.writeTo(TABLE).overwrite(F.col("data_date") == F.lit(BATCH_DATE))

def write_log(started, status, stage, count, error_type, landing_run):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.pipeline")
    # Không log exception message/response vì có thể chứa thông tin cá nhân hoặc token.
    message = json.dumps({"data_date": ETL_Date, "run_id": RUN_ID, "landing_run_id": landing_run,
                          "stage": stage, "error_type": error_type}, ensure_ascii=False)
    schema = "target_table string, source_table string, log_message string, status string, start_date timestamp, end_date timestamp, step string, record_count long"
    df_log = spark.createDataFrame([(TABLE, SOURCE, message, status, started,
                                    datetime.now(timezone.utc), STEP, count)], schema)
    ddl = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in df_log.schema)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {LOG_TABLE} ({ddl}) USING iceberg")
    df_log.writeTo(LOG_TABLE).append()


def write_to_iceberg(spark_session, df_data, iceberg_full_tableName, ETL_Date):
    # Same function signature as sample; atomic overwrite instead of DELETE+APPEND.
    if spark_session is not spark or iceberg_full_tableName != TABLE:
        raise ValueError("Spark/table không khớp cấu hình job")
    write_bronze(df_data)


In [ ]:
globals().pop('BRONZE_RESULT', None)
RUN_ID = uuid.uuid4().hex
started = datetime.now(timezone.utc)
stage, count, landing_run = "landing_read", 0, None
committed = False
TABLE = TABLE_NAME
CATALOG = ICEBERG_CATALOG1
logger.info("START Landing -> Bronze table=%s date=%s run_id=%s", TABLE, ETL_Date, RUN_ID)
s3 = None
try:
    s3 = init_landing()
    spark = init_spark()
    spark.conf.set("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh")
    manifest = resolve_manifest()
    landing_run = manifest["run_id"]
    rows = load_landing(manifest)
    count = len(rows)
    logger.info("Landing validated rows=%s landing_run_id=%s", count, landing_run)
    stage = "bronze_write"
    df = bronze_frame(rows, manifest)
    write_to_iceberg(spark, df, TABLE_NAME, ETL_Date)
    committed, stage = True, "bronze_committed"
    logger.info("BRONZE COMMITTED table=%s rows=%s", TABLE, count)
    write_log(started, "Success", stage, count, None, landing_run)
    logger.info("SUCCESS Bronze và log đã ghi xong")
    print("TABLE =", TABLE)
    print("RUN_ID =", RUN_ID)
    print("LANDING_RUN_ID =", landing_run)
    status = "Success"
    BRONZE_RESULT = {"status": "Success", "count": count, "run_id": RUN_ID, "landing_run_id": landing_run}
except Exception as exc:
    status = "Fail"
    error = error_summary(exc)
    log_message = error
    logger.error("FAIL stage=%s committed=%s error=%s", stage, committed, error)
    if not committed and "spark" in globals():
        try:
            write_log(started, "Fail", stage, count, type(exc).__name__, landing_run)
        except Exception as log_exc:
            logger.error("Ghi log Iceberg cũng lỗi: %s", type(log_exc).__name__)
    try:
        put_json(f"{LOG_PREFIX}/bronze-{RUN_ID}.json", {
            "status": "Fail", "stage": stage, "bronze_committed": committed,
            "run_id": RUN_ID, "landing_run_id": landing_run,
            "data_date": ETL_Date, "error": error, "record_count": count})
    except Exception as log_exc:
        logger.error("Không ghi được log S3: %s", type(log_exc).__name__)
    raise RuntimeError(f"Bronze thất bại stage={stage}; bronze_committed={committed}; {error}") from exc


finally:
    end_date = datetime.now(timezone.utc)
    record_count = count
    logger.info("END status=%s rows=%s stage=%s bronze_committed=%s", status, record_count, stage, committed)


In [ ]:
# Kiểm tra số dòng; không gọi API, không in thông tin nhân sự.
batch = spark.table(TABLE).filter(F.col("data_date") == F.lit(BATCH_DATE))
batch.groupBy("data_date", "run_id", "landing_run_id").count().show(truncate=False)
if "BRONZE_RESULT" in globals():
    actual = batch.count()
    print("Số dòng thực tế:", actual, "| kỳ vọng:", BRONZE_RESULT["count"])
    if actual != BRONZE_RESULT["count"]:
        raise ValueError("Số dòng Bronze không khớp kết quả vừa ghi")
spark.table(LOG_TABLE).filter(F.col("log_message").contains(RUN_ID)).show(10, truncate=False)


## Vận hành
1. Import bằng Local file. Landing dùng Python với requests/boto3; Bronze dùng Python with Spark với connector như mẫu.
2. Restart kernel và Run All để thay thế các hàm cũ còn trong kernel. Truyền cùng ETL_Date cho hai notebook.
3. API đơn vị dùng request mẫu với access_token/date_start/date_end; không cần confirm. API nhân sự cần tên tham số request phân trang thực tế trong HRM_PAGE_PARAM và HRM_SIZE_PARAM. Kiểm tra total/số trang bằng response thực tế, không dùng cờ xác nhận.
4. Landing lấy và kiểm tra xong API; xóa tập file cũ đúng `hrm/<entity>/YYYYMMDD/`; ghi trang mới và ghi manifest cuối cùng. Nếu API lỗi trước bước xóa, tập cũ được giữ. Nếu lỗi trong lúc ghi file, không có manifest hoàn tất.
5. Bronze tự đọc `manifest.json` theo entity/ETL_Date; không cần copy đường dẫn. Để MANIFEST_PATH_CONFIG và LANDING_RUN_ID_CONFIG trống nếu muốn đọc tập hiện hành của ngày. Nếu truyền tùy chọn này, giá trị phải khớp manifest hiện tại.
6. Bronze overwrite atomic partition data_date=ETL_Date, tương đương thay thế toàn bộ dữ liệu ngày mà không có khoảng DELETE/APPEND. API success với 0 bản ghi vẫn thay thế ngày bằng tập rỗng; request lỗi không được coi là rỗng.
7. Chạy Landing xong mới chạy Bronze; không chạy lại Landing cùng ngày khi Bronze đang đọc. S3 không có atomic replace toàn thư mục; checksum phát hiện nội dung thay đổi giữa lúc đọc. Khi Landing đã chạy lại, replay sẽ dùng tập ngày mới, không còn tập run cũ.
8. Log Landing/backup log Bronze nằm ngoài dữ liệu ngày: `hrm/<entity>/_logs/YYYYMMDD/`. Log Bronze giữ schema bronze.pipeline.log_pipeline như mẫu. Cell cuối kiểm tra số lượng/manifest, không in dữ liệu nhân sự.
Giới hạn: tập dữ liệu ngày được giữ trong RAM driver; chưa khắc phục lỗi metastore của runtime. Các trường hợp total thay đổi hoặc lặp trang/bản ghi vẫn báo lỗi để tránh chốt dữ liệu thiếu/trùng.
